---
# <p align="center"> CS5920 Assessed Coursework Assignment 1 </p>
---

## Library Imports

In [1]:
import numpy as np
from numpy import inf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

random_state = 1301  

## 1. Loading Datasets

In [2]:
# Load iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Load ionosphere dataset
ionosphere_data = np.genfromtxt('ionosphere.txt', delimiter=',')
X_ionosphere = ionosphere_data[:, :-1]
y_ionosphere = ionosphere_data[:, -1]

print("Iris dataset loaded:")
print(f"Features shape: {X_iris.shape}")
print(f"Labels shape: {y_iris.shape}")
print(f"Unique labels: {np.unique(y_iris)}")
print("\nIonosphere dataset loaded:")
print(f"Features shape: {X_ionosphere.shape}")
print(f"Labels shape: {y_ionosphere.shape}")
print(f"Unique labels: {np.unique(y_ionosphere)}")

Iris dataset loaded:
Features shape: (150, 4)
Labels shape: (150,)
Unique labels: [0 1 2]

Ionosphere dataset loaded:
Features shape: (351, 34)
Labels shape: (351,)
Unique labels: [-1.  1.]


## 2. Splitting Datasets into Training and Test Sets

In [3]:
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(X_iris, y_iris, random_state=1301)

X_ionosphere_train, X_ionosphere_test, y_ionosphere_train, y_ionosphere_test = train_test_split(X_ionosphere, y_ionosphere, random_state=1301)

print('spliting dataset into training and test sets')
print(f'\nIris - Training: {X_iris_train.shape[0]}, Test: {X_iris_test.shape[0]}')
print(f'Ionosphere - Training: {X_ionosphere_train.shape[0]}, Test: {X_ionosphere_test.shape[0]}')

spliting dataset into training and test sets

Iris - Training: 112, Test: 38
Ionosphere - Training: 263, Test: 88


## 3. Helper Functions

In [4]:
def euclidean_distance(x1, x2):
    distance = 0.0
    for i in range(len(x1)):
        diff = x1[i] - x2[i]
        distance += diff * diff
    return np.sqrt(distance)

## 4. Nearest Neighbor Algorithm Implementation

In [5]:
class NearestNeighbor:
    def __init__(self):
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def predict(self, X_test):
        predictions = []
        
        for test_sample in X_test:
            min_distance = float('inf')
            nearest_label = None
            
            for i in range(len(self.X_train)):
                dist = euclidean_distance(test_sample, self.X_train[i])
                if dist < min_distance:
                    min_distance = dist
                    nearest_label = self.y_train[i]
            
            predictions.append(nearest_label)
        
        return np.array(predictions)
    
    def calculate_error_rate(self, X_test, y_test):
        predictions = self.predict(X_test)
        errors = np.sum(predictions != y_test)
        error_rate = errors / len(y_test)
        return errors, error_rate

## 5. Apply NN to Datasets

In [6]:
# 1-NN for Iris dataset
nn_iris = NearestNeighbor()
nn_iris.fit(X_iris_train, y_iris_train)
errors_iris, error_rate_iris = nn_iris.calculate_error_rate(X_iris_test, y_iris_test)

print(f"Number of errors: {errors_iris}")
print(f"Test error rate: {error_rate_iris:.6f}")

Number of errors: 1
Test error rate: 0.026316


In [7]:
#1-NN for Ionosphere dataset
nn_ionosphere = NearestNeighbor()
nn_ionosphere.fit(X_ionosphere_train, y_ionosphere_train)
errors_ionosphere, error_rate_ionosphere = nn_ionosphere.calculate_error_rate(X_ionosphere_test, y_ionosphere_test)

print(f"Number of errors: {errors_ionosphere}")
print(f"Test error rate: {error_rate_ionosphere:.6f}")

Number of errors: 11
Test error rate: 0.125000


## 6. Conformal Predictor Implementation based on NN

In [8]:
def conformity_score(x, y, X_train, y_train):
    min_dist_same = inf
    min_dist_diff = inf
    
    for i in range(len(X_train)):
        dist = euclidean_distance(x, X_train[i])
        
        if y_train[i] == y:
            if dist < min_dist_same:
                min_dist_same = dist
        else:
            if dist < min_dist_diff:
                min_dist_diff = dist
    
    # Case 0/0: Define as 0 (standard convention)
    if min_dist_same == 0 and min_dist_diff == 0:
        return 0.0
    # Case a/0 where a > 0: Define as infinity (as mentioned in assignment)
    elif min_dist_same == 0:
        return inf
    # no condition
    else:
        return min_dist_diff / min_dist_same

In [9]:
def conformal_predictor(X_train, y_train, X_test, y_test):
    unique_labels = []
    for label in y_train:
        if label not in unique_labels:
            unique_labels.append(label)
    
    p_values_all = []
    
    for test_idx in range(len(X_test)):
        x_test = X_test[test_idx]
        p_values_for_sample = []
        
        for assumed_label in unique_labels:
            test_conformity = conformity_score(x_test, assumed_label, X_train, y_train)
            
            train_conformities = []
            for i in range(len(X_train)):
                conf_score = conformity_score(X_train[i], y_train[i], X_train, y_train)
                train_conformities.append(conf_score)
            
            count = 0
            for conf_score in train_conformities:
                if conf_score <= test_conformity:
                    count += 1
    
            count += 1
            p_value = count / (len(train_conformities) + 1)
            p_values_for_sample.append(p_value)
        
        p_values_all.append(p_values_for_sample)
    
    return np.array(p_values_all), unique_labels

In [10]:
def calculate_average_false_p_value(p_values, y_test, unique_labels):
   
    false_p_values = []
    
    for test_idx in range(len(y_test)):
        true_label = y_test[test_idx]
        
        for label_idx, label in enumerate(unique_labels):
            if label != true_label:
                false_p_values.append(p_values[test_idx][label_idx])
    
    return np.mean(false_p_values)

## 7. Apply Conformal Predictor to Datasets

In [11]:
# Conformity for iris dataset
p_values_iris, labels_iris = conformal_predictor(X_iris_train, y_iris_train, X_iris_test, y_iris_test)
avg_false_p_iris = calculate_average_false_p_value(p_values_iris, y_iris_test, labels_iris)

print("\nConformal Predictor Results for Iris Dataset:")
print(f"Average false p-value: {avg_false_p_iris:.6f}")


Conformal Predictor Results for Iris Dataset:
Average false p-value: 0.008850


In [12]:
#Conformity for ionosphere dataset
p_values_ionosphere, labels_ionosphere = conformal_predictor(X_ionosphere_train, y_ionosphere_train, X_ionosphere_test, y_ionosphere_test)
avg_false_p_ionosphere = calculate_average_false_p_value(p_values_ionosphere, y_ionosphere_test, labels_ionosphere)

print("\nConformal Predictor Results for Ionosphere Dataset:")
print(f"Average false p-value: {avg_false_p_ionosphere:.6f}")


Conformal Predictor Results for Ionosphere Dataset:
Average false p-value: 0.003788


## Final Results as asked in coursework

In [13]:
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*80 + "\n")

print("1. NN(1-NN) - Test Error Rates**\n")
print(f"   • Iris Dataset: {error_rate_iris:.6f}")
print(f"   • Ionosphere Dataset: {error_rate_ionosphere:.6f}")

print("\n2. Conformality  - Average False P-Values**\n")
print(f"   • Iris Dataset: {avg_false_p_iris:.6f}")
print(f"   • Ionosphere Dataset: {avg_false_p_ionosphere:.6f}")

print("\n" + "="*60)



FINAL RESULTS SUMMARY

1. NN(1-NN) - Test Error Rates**

   • Iris Dataset: 0.026316
   • Ionosphere Dataset: 0.125000

2. Conformality  - Average False P-Values**

   • Iris Dataset: 0.008850
   • Ionosphere Dataset: 0.003788

